# Тема 3: Предобработка данных для ML (ОТВЕТЫ)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

df = pd.merge(pd.read_csv('data/transactions.csv'), pd.read_csv('data/customers.csv'), on='customer_id')
df = pd.merge(df, pd.read_csv('data/products.csv'), on='product_id')
df['total_sum'] = df['price'] * df['quantity']
df['age'] = df['age'].fillna(df['age'].median())

### Задание 1

In [ ]:
print(df.dtypes)
# Числовые: age, price, quantity, total_sum
# Категориальные: gender, location, category

### Задание 2

In [ ]:
le = LabelEncoder()
df['gender_encoded'] = le.fit_transform(df['gender'])
print(df[['gender', 'gender_encoded']].drop_duplicates())

### Задание 3

In [ ]:
df = pd.get_dummies(df, columns=['location'], drop_first=True)
# One-Hot лучше, потому что у городов нет порядка (Москва не «больше» Казани)
df.head()

### Задание 4

In [ ]:
df = pd.get_dummies(df, columns=['category'], drop_first=True)
# drop_first убирает одну колонку для избежания мультиколлинеарности
df.head()

### Задание 5

In [ ]:
scaler = StandardScaler()
df[['age_scaled','price_scaled','quantity_scaled']] = scaler.fit_transform(df[['age','price','quantity']])
df[['age_scaled','price_scaled','quantity_scaled']].describe()

### Задание 6

In [ ]:
mm_scaler = MinMaxScaler()
df[['age_mm','price_mm','quantity_mm']] = mm_scaler.fit_transform(df[['age','price','quantity']])
df[['age_mm','price_mm','quantity_mm']].describe()

### Задание 7

In [ ]:
feature_cols = [c for c in df.columns if c not in ['transaction_id','customer_id','product_id','date','signup_date','gender','total_sum']]
X = df[feature_cols].select_dtypes(include=[np.number])
y = df['total_sum']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

### Задание 8

In [ ]:
df['is_high_spender'] = (df['total_sum'] > df['total_sum'].median()).astype(int)
print(df['is_high_spender'].value_counts(normalize=True))

### Задание 9

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['is_weekend'] = (df['date'].dt.dayofweek >= 5).astype(int)
df['age_group'] = pd.cut(df['age'], bins=[0,25,45,100], labels=[0,1,2]).astype(int)
df[['is_weekend','age_group']].head()

### Задание 10

In [ ]:
df_clean = df.drop(columns=['transaction_id','customer_id','product_id','date','signup_date'], errors='ignore')
# ID — это уникальные идентификаторы, модель запомнит их и переобучится
df_clean.shape

### Задание 11

In [ ]:
X_final = df_clean.select_dtypes(include=[np.number]).drop(columns=['total_sum','is_high_spender'], errors='ignore')
print(X_final.shape)
print(X_final.dtypes)
X_final.head()

### Задание 12

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
X_num = df[['age','price','quantity']]
y = df['total_sum']
X_tr, X_te, y_tr, y_te = train_test_split(X_num, y, test_size=0.2, random_state=42)
pipe.fit(X_tr, y_tr)
print('R2:', pipe.score(X_te, y_te))